In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor
from glob import glob

import pandas as pd

DATASET_NAME = "ContentWise_LastActionSplit"

In [ ]:
results_dir = f"data/hyperparameter_optimization_results/{DATASET_NAME}"
json_paths = glob(f"{results_dir}/*/results.json")
if not json_paths:
    raise ValueError(f"No results found in {results_dir}")


def read_json_file(path):
    try:
        with open(path, "rb") as f:
            return json.load(f)
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None


with ThreadPoolExecutor(max_workers=32) as executor:
    results = [result for result in executor.map(read_json_file, json_paths) if result is not None]
print(f"Loaded {len(results)} results for {DATASET_NAME}")

df = pd.json_normalize(results)
df["start_timestamp"] = pd.to_datetime(df["start_timestamp"], format="%Y-%m-%d_%H-%M-%S_%f")
df["end_timestamp"] = pd.to_datetime(df["end_timestamp"], format="%Y-%m-%d_%H-%M-%S_%f")
df["duration"] = (df["end_timestamp"] - df["start_timestamp"]).dt.total_seconds() / 60
df.groupby("model_name").size()

In [ ]:
params_per_model = {}
commands_per_model = {}

for model_name in df["model_name"].unique():
    with open(f"data/hyperparameter_grids/{DATASET_NAME}/commands_{model_name}.txt", "r") as f:
        command = f.readline().strip()
    params_per_model[model_name] = [part[2:].split("=")[0] for part in command.split(" ") if part.startswith("--")]
    commands_per_model[model_name] = command

In [ ]:
IGNORED_COLUMNS = ["hyperparams.run_id", "hyperparams.item_features_numpy", "hyperparams.results_dir"]
NUM_EVAL_RUNS = 20 if "_synthetic_" in DATASET_NAME else 100

command_dir = f"data/evaluation_commands/{DATASET_NAME}"
os.makedirs(command_dir, exist_ok=True)

for model_name in df["model_name"].unique():
    model_df = df[df["model_name"] == model_name].copy()

    group_cols = ["model_name"] + [
        col for col in model_df.columns
        if "hyperparams" in col
        and col not in IGNORED_COLUMNS + ["hyperparams.seed"]
        and len(model_df[col].unique()) > 1
    ]
    model_df[group_cols] = model_df[group_cols].fillna(0)

    grouped = (
        model_df[group_cols + ["results.nll", "results.ndcg", "results.slate_nll", "duration"]]
        .groupby(group_cols)
        .agg(
            mean_nll=("results.nll", "mean"),
            count=("results.nll", "size"),
            mean_ndcg=("results.ndcg", "mean"),
            mean_slate_nll=("results.slate_nll", "mean"),
            mean_duration_minutes=("duration", "mean"),
        )
        .sort_values("mean_nll", ascending=True)
    )
    print(f"{model_name}: {model_df.shape[0]} runs")
    display(grouped.head(5).round(4))

    best_params = grouped.reset_index().iloc[0].to_dict()

    cmd = commands_per_model[model_name]
    for param in params_per_model[model_name]:
        param = param.replace("no-", "")

        if param == "model_name" or ('hyperparams.' + param) not in best_params:
            continue

        if f"--{param}=" in cmd:
            first_part = cmd.split(f"--{param}=")[0]

            if "--" in cmd.split(f"--{param}=")[1]:
                second_part = " --" + "--".join(cmd.split(f"--{param}=")[1].split("--")[1:])
            else:
                second_part = ""

            center_part = f"--{param}={best_params['hyperparams.' + param]}"


            cmd = f"{first_part} {center_part}{second_part}"
        elif f"--{param}" in cmd:
            if best_params['hyperparams.' + param] is True:
                center_part = f"--{param}"
            else:
                center_part = f"--no-{param}"

            cmd = cmd.replace(f"--{param}", f"{center_part}")
        elif f"--no-{param}" in cmd:
            if best_params['hyperparams.' + param] is True:
                center_part = f"--{param}"
            else:
                center_part = f"--no-{param}"

            cmd = cmd.replace(f"--no-{param}", f"{center_part}")
        else:
            print(f"Parameter --{param}= not found in command.")

    if DATASET_NAME != "Stockholm_LastActionSplit":
        cmd = cmd.replace("--no-store_trained_model", "--store_trained_model")
    else:
        cmd = cmd.replace("--store_trained_model", "--no-store_trained_model")
    cmd = cmd.replace("--optimize_hyperparams", "--no-optimize_hyperparams")

    if model_name == "Intercept" and "--no-disable_cpu_fallback" not in cmd:
        cmd = cmd + " --no-disable_cpu_fallback"

    with open(f"{command_dir}/commands_{model_name}.txt", "w", newline="\n") as f:
        for run in range(1, NUM_EVAL_RUNS + 1):
            f.write(cmd.replace("--seed=", f"--seed={run}") + "\n")

if os.path.exists(f"{command_dir}/commands_Intercept.txt"):
    with open(f"{command_dir}/commands_Intercept.txt", "r") as f:
        intercept_commands = f.readlines()
    for model_name in ["Random", "MostPop"]:
        with open(f"{command_dir}/commands_{model_name}.txt", "w") as f:
            for cmd in intercept_commands:
                f.write(cmd.replace("--model_name=Intercept", f"--model_name={model_name}"))